In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 

In [2]:
df = pd.read_csv('telecom_customer_churn_realistic_100k.csv')

In [3]:
df.head()

,Customer_ID,Gender,Age,Senior_Citizen,Marital_Status,Region,Tenure,Contract_Type,Internet_Service,Phone_Service,...,Monthly_Charges,Total_Charges,Data_Usage_GB,Call_Minutes,Support_Tickets,Satisfaction_Score,Last_Login_Days,Late_Payments,Auto_Pay,Churn
0,CUST000001,Male,39,Yes,Divorced,South,7,Two Year,NaN,Yes,...,844.67,5750.17,45.2,355,0,6.0,8,0,No,Yes
1,CUST000002,Female,33,Yes,Divorced,North,10,Month-to-Month,Fiber,Yes,...,891.31,9044.86,238.8,219,1,7.0,9,2,Yes,No
2,CUST000003,Male,59,Yes,Married,West,18,Two Year,5G,Yes,...,1097.05,19857.30,196.8,248,0,5.0,36,0,No,No
3,CUST000004,Male,30,No,Divorced,North,54,Month-to-Month,NaN,No,...,718.32,38650.42,188.8,329,12,7.0,19,1,Yes,No
4,CUST000005,Male,18,No,Married,North,50,Two Year,Fiber,No,...,1127.51,56480.50,179.8,363,0,5.0,1,1,Yes,No


In [4]:
df.isnull().mean()*100

Customer_ID            0.000
Gender                 0.000
Age                    0.000
Senior_Citizen         0.000
Marital_Status         0.000
Region                 0.000
Tenure                 0.000
Contract_Type          0.000
Internet_Service      25.047
Phone_Service          0.000
Multiple_Lines         0.000
Tech_Support           8.065
Streaming_Service      0.000
Payment_Method         3.083
Monthly_Charges        2.046
Total_Charges          4.981
Data_Usage_GB          0.000
Call_Minutes           0.000
Support_Tickets        0.000
Satisfaction_Score     4.007
Last_Login_Days        0.000
Late_Payments          0.000
Auto_Pay               0.000
Churn                  0.000
dtype: float64

# **Investigate why values are missing**

In [5]:
df['Internet_Service'].value_counts()

Internet_Service
Fiber    25289
5G       25079
DSL      24585
Name: count, dtype: int64

In [6]:
df["Internet_Service_missing"] = df["Internet_Service"].isnull().astype(int)

In [7]:
categorical_columns = df.select_dtypes(('object','str'))

In [8]:
for feature in categorical_columns:
    print(f"\n--- {feature} ---")
    
    result = pd.crosstab(
        df[feature],
        df['Internet_Service_missing'],
        normalize="columns"
    ) * 100
    
    print(result)



--- Customer_ID ---
Internet_Service_missing         0         1
Customer_ID                                 
CUST000001                0.000000  0.003992
CUST000002                0.001334  0.000000
CUST000003                0.001334  0.000000
CUST000004                0.000000  0.003992
CUST000005                0.001334  0.000000
...                            ...       ...
CUST099996                0.001334  0.000000
CUST099997                0.001334  0.000000
CUST099998                0.001334  0.000000
CUST099999                0.000000  0.003992
CUST100000                0.001334  0.000000

[100000 rows x 2 columns]

--- Gender ---
Internet_Service_missing          0          1
Gender                                        
Female                    50.074046  49.910169
Male                      49.925954  50.089831

--- Senior_Citizen ---
Internet_Service_missing          0          1
Senior_Citizen                                
No                        49.547049  50.02595


### Main observation

* **Gender:** ~50/50 in both groups → almost no difference.
* **Senior Citizen:** ~50/50 → very little difference.
* **Marital Status:** ~33/33/33 → almost identical.
* **Region:** each region ~20% → practically identical.
* **Contract Type:** ~55%, 25%, 20% → almost identical.
* **Phone Service:** ~50/50 → no meaningful difference.
* **Multiple Lines:** ~50/50 → no meaningful difference.
* **Tech Support:** ~50/50 → very small difference.
* **Streaming Service:** ~50/50 → very small difference.
* **Payment Method:** each ~20% → almost identical.
* **Auto Pay:** ~50/50 → almost identical.
* **Churn:**

  * Missing = 0 → **24.79% churn**
  * Missing = 1 → **25.22% churn**
  * Difference ≈ **0.43 percentage points**, which is very small.



In [9]:
numerical_columns = df.select_dtypes(('int64','float64'))
for feature in numerical_columns:
    print(f"\n--- {feature} ---")
    
    result = df.groupby('Internet_Service_missing')[feature].agg(
        ['count', 'mean', 'median', 'std', 'min', 'max']
    ).round(2)
    
    print(result)


--- Age ---
                          count   mean  median    std  min  max
Internet_Service_missing                                       
0                         74953  41.80    42.0  13.40   18  106
1                         25047  41.86    42.0  13.54   18   95

--- Tenure ---
                          count   mean  median    std  min  max
Internet_Service_missing                                       
0                         74953  23.53    16.0  24.16    0  255
1                         25047  23.53    16.0  24.05    0  228

--- Monthly_Charges ---
                          count    mean  median     std   min      max
Internet_Service_missing                                              
0                         73401  900.63  900.31  319.03  99.0  2175.26
1                         24553  902.01  900.46  319.70  99.0  2195.64

--- Total_Charges ---
                          count      mean    median       std    min  \
Internet_Service_missing                               

In [10]:
import pandas as pd
from scipy.stats import chi2_contingency

# Get categorical columns
cat_cols = df.select_dtypes(include=["object", "category"]).columns

# Remove target/missing column and ID
cat_cols = [
    col for col in cat_cols
    if col not in ["Internet_Service", "Customer_ID"]
]

for col in cat_cols:

    contingency = pd.crosstab(
        df[col],
        df["Internet_Service_missing"]
    )

    chi2, p, dof, expected = chi2_contingency(contingency)

    print("=" * 60)
    print(f"Column   : {col}")
    print(f"P-value  : {p:.4f}")

    if p < 0.05:
        print("Missingness is associated with this variable")
        print("(Evidence against MCAR based on this variable)")
    else:
        print("No evidence that missingness depends on this variable")

C:\Users\md salman\AppData\Local\Temp\ipykernel_12972\4292311926.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=["object", "category"]).columns


Column   : Gender
P-value  : 0.6586
No evidence that missingness depends on this variable
Column   : Senior_Citizen
P-value  : 0.1919
No evidence that missingness depends on this variable
Column   : Marital_Status
P-value  : 0.3057
No evidence that missingness depends on this variable
Column   : Region
P-value  : 0.6926
No evidence that missingness depends on this variable
Column   : Contract_Type
P-value  : 0.6901
No evidence that missingness depends on this variable
Column   : Phone_Service
P-value  : 0.4502
No evidence that missingness depends on this variable
Column   : Multiple_Lines
P-value  : 0.4824
No evidence that missingness depends on this variable
Column   : Tech_Support
P-value  : 0.2206
No evidence that missingness depends on this variable
Column   : Streaming_Service
P-value  : 0.6016
No evidence that missingness depends on this variable
Column   : Payment_Method
P-value  : 0.3558
No evidence that missingness depends on this variable
Column   : Auto_Pay
P-value  : 0.4388

In [13]:
from scipy.stats import mannwhitneyu

for col in numerical_columns:

    group_0 = df.loc[
        df['Internet_Service_missing'] == 0, col
    ].dropna()

    group_1 = df.loc[
        df['Internet_Service_missing'] == 1, col
    ].dropna()

    stat, p = mannwhitneyu(
        group_0,
        group_1,
        alternative='two-sided'
    )

    print("=" * 60)
    print(f"Column  : {col}")
    print(f"P-value : {p:.4f}")

    if p < 0.05:
        print("Significant difference in distributions")
    else:
        print("No significant difference")

Column  : Age
P-value : 0.6418
No significant difference
Column  : Tenure
P-value : 0.7806
No significant difference
Column  : Monthly_Charges
P-value : 0.6656
No significant difference
Column  : Total_Charges
P-value : 0.4357
No significant difference
Column  : Data_Usage_GB
P-value : 0.9045
No significant difference
Column  : Call_Minutes
P-value : 0.9486
No significant difference
Column  : Support_Tickets
P-value : 0.9491
No significant difference
Column  : Satisfaction_Score
P-value : 0.7677
No significant difference
Column  : Last_Login_Days
P-value : 0.5729
No significant difference
Column  : Late_Payments
P-value : 0.7238
No significant difference
Column  : Internet_Service_missing
P-value : 0.0000
Significant difference in distributions


### Observation:
No statistically significant association was found between Internet_Service missingness and any of the analyzed categorical or numerical variables. This suggests that the missingness does not appear to be systematically related to the observed features. Therefore, there is no evidence against MCAR based on the variables tested.

## Checking Inconsistent values

In [14]:
for col in numerical_columns:
    print(f"\n--- {col} ---")
    print("Min:", df[col].min())
    print("Max:", df[col].max())
    print("Unique values:", df[col].nunique())


--- Age ---
Min: 18
Max: 106
Unique values: 82

--- Tenure ---
Min: 0
Max: 255
Unique values: 213

--- Monthly_Charges ---
Min: 99.0
Max: 2195.64
Unique values: 66236

--- Total_Charges ---
Min: -89.62
Max: 367024.79
Unique values: 93702

--- Data_Usage_GB ---
Min: 0.0
Max: 575.8
Unique values: 4446

--- Call_Minutes ---
Min: 0
Max: 1111
Unique values: 964

--- Support_Tickets ---
Min: 0
Max: 22
Unique values: 23

--- Satisfaction_Score ---
Min: 1.0
Max: 10.0
Unique values: 10

--- Last_Login_Days ---
Min: 0
Max: 170
Unique values: 140

--- Late_Payments ---
Min: 0
Max: 16
Unique values: 16

--- Internet_Service_missing ---
Min: 0
Max: 1
Unique values: 2


In [20]:
(df['Total_Charges'] < 0).sum()

np.int64(21)

In [24]:
df[df['Total_Charges'] < 0][
    ['Customer_ID', 'Total_Charges', 'Tenure', 'Monthly_Charges']
]

,Customer_ID,Total_Charges,Tenure,Monthly_Charges


In [22]:
df.loc[df['Total_Charges'] < 0, 'Total_Charges'].describe()

count    21.000000
mean    -38.980476
std      24.896119
min     -89.620000
25%     -50.040000
50%     -34.990000
75%     -21.400000
max      -0.520000
Name: Total_Charges, dtype: float64

In [23]:
df.loc[df['Total_Charges'] < 0, 'Total_Charges'] = np.nan